# Practical Project for Machine Learning Algorithms Course

## Predicting Mexico City’s Airbnb Rental Prices

The problem under study consists of predicting short-term rental prices for Airbnb listings in Mexico City, Mexico. More specifically, the goal is to estimate the nightly rental price of a listing based on its observable characteristics, such as location, type of accommodation, and available amenities.

### Objectives

- Explore and understand the dataset characteristics
- Prepare and clean data for model training
- Select and justify appropriate machine learning algorithms
- Implement and experiment with different model configurations
- Evaluate and validate model performance
- Prepare model for scalability and deployment considerations

### Dataset

The dataset contains information about Airbnb listings in Mexico City, including property characteristics, location data, host information, and pricing details.

*Data source: [Inside Airbnb](https://data.insideairbnb.com/mexico/df/mexico-city/2025-06-25/data/listings.csv.gz)*

---

*Developed by Nuno Miguel Carvalho Araújo, No. 20078*

### 1. Environment Preparation

This section establishes the computational environment required for the analysis.

#### 1.1. Library Imports

Required packages for data handling, machine learning algorithms, and visualization tools are imported to support the complete analytical workflow.

In [ ]:
# Default imports
import os
import requests
import ast
import time

# Pandas imports
import pandas as pd

# Numpy imports
import numpy as np

# Define global plotting configuration
import matplotlib.pyplot as plt
import seaborn as sns

# Collections imports
from collections import Counter

# Sklearn imports
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

#### 1.2. Visualization Configuration

Global plotting parameters and color schemes are configured to ensure consistent and professional visualizations throughout the analysis.

In [ ]:
# Set consistent style
plt.style.use('default')
sns.set_palette("husl")

# Define color scheme
COLORS = {
    'primary': '#1f77b4',
    'secondary': '#ff7f0e',
    'bg': '#f0f0f0',
    'heatmap': 'viridis'
}

# Set default figure parameters
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 10

### 2. Data Comprehension

This section focuses on loading and exploring the dataset to understand its structure, features, and statistical properties before proceeding to data preparation.

#### 2.1. Download and Save Dataset

Download the dataset from remote storage and save it locally for analysis.

In [ ]:
# Download url
download_url = "https://drive.usercontent.google.com/download?id=1mwtqAe3NzdyusHpUtkYwQzNtEAeSFI_a&export=download&confirm=t"

# Target path to save the csv file
target_csv_path = "datasets/mc-full-listings.csv"

# Check if the file already exists before download it
if (not os.path.exists(target_csv_path)):
    # Create a local file with remote csv data
    response = requests.get(download_url)
    response.raise_for_status()

    # Write the content to a local file
    with open(target_csv_path, "wb") as f:
        f.write(response.content)

    # Confirmation message
    print("File downloaded successfully.")
else:
    # Confirmation message
    print("File already exists. Skipping download.")

#### 2.2. Load Dataset

Load the dataset into a pandas DataFrame and display basic information about its shape and content.

In [ ]:
# Load dataset as a DataFrame
df = pd.read_csv("datasets/mc-full-listings.csv")

# Display DataFrame shape
print(f"The dataset contains {df.shape[0]} listings and {df.shape[1]} features.")

# Display first few rows of the DataFrame
display(df.head())

#### 2.3. Explore Feature Types and Structure

Analyze the dataset structure to identify feature types, missing values, and data quality issues.

In [ ]:
# Display DataFrame info
df.info()

*The dataset requires preprocessing. Price will be converted to float for numerical analysis. Additional transformations will be applied in the next section.*

##### 2.3.1. Convert Price to Numeric Format

The price feature is stored as a string with currency symbols. Convert it to float for numerical analysis.

In [ ]:
# Convert price from string to float
df['price'] = df['price'].str.replace('$', '').str.replace(',', '').astype(float)

# Verify conversion
print("Price data type after conversion:", df['price'].dtype)
print(f"Price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")

##### 2.3.2. Statistical Summary of Numeric Features

Display descriptive statistics to understand the distribution and range of numerical features.

In [ ]:
# Display DataFrame summary statistics
display(df.describe())

##### 2.3.3. Correlation Between Numeric Features

Visualize correlations between all numeric features to identify potential relationships with the target variable (price).

In [ ]:
# Correlation matrix for numeric features
numeric_df = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()

plt.figure(figsize=(12, 10))
plt.imshow(correlation_matrix, cmap=COLORS['heatmap'], aspect='auto', vmin=-1, vmax=1)
plt.colorbar(label='Correlation Coefficient')
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=90)
plt.yticks(range(len(correlation_matrix.columns)), correlation_matrix.columns)
plt.title('Correlation Matrix - All Numeric Features')
plt.tight_layout()
plt.show()

*The correlation matrix reveals moderate relationships between features, though no strong correlations stand out significantly.*

##### 2.3.4. Distribution of Key Categorical Features

Analyze the most common categories for room type, property type, and neighbourhood to understand the dataset composition.

In [ ]:
# Distribution of key categorical features
categorical_features = ['room_type', 'property_type', 'neighbourhood_cleansed']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(categorical_features):
    if feature in df.columns:
        top_values = df[feature].value_counts().head(10)
        axes[idx].barh(range(len(top_values)), top_values.values, color=COLORS['primary'])
        axes[idx].set_yticks(range(len(top_values)))
        axes[idx].set_yticklabels(top_values.index)
        axes[idx].set_xlabel('Count')
        axes[idx].set_title(f'Top 10 {feature}')
        axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

*Most listings are entire homes/apartments, with "Entire rental unit" as the dominant property type and Cuauhtémoc as the most common neighbourhood.*

##### 2.3.5. Geographic Distribution of Listings

Visualize the spatial distribution of listings across Mexico City and their relationship with price.

In [ ]:
# Geographic distribution of listings
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
# Density map
hexbin = axes[0].hexbin(df['longitude'], df['latitude'], gridsize=50, cmap=COLORS['heatmap'], mincnt=1, edgecolors='face', linewidths=0.2)
axes[0].set_facecolor(COLORS['bg'])
plt.colorbar(hexbin, ax=axes[0], label='Number of Listings')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('Density of Listings in Mexico City')
    
# Price map with better visibility
scatter = axes[1].scatter(df['longitude'], df['latitude'], c=df['price'], cmap=COLORS['heatmap'], alpha=0.6, s=15, vmax=df['price'].quantile(0.95), edgecolors='black', linewidths=0.1)
axes[1].set_facecolor(COLORS['bg'])
plt.colorbar(scatter, ax=axes[1], label='Price ($)')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('Geographic Distribution by Price')
    
plt.tight_layout()
plt.show()

*Listings are geographically concentrated in the central area (latitude: 19.20-19.50, longitude: -99.30 to -99.00), with notable price variations across different locations.*

#### 2.4. Target Variable Analysis

Perform detailed analysis of the target variable (price) to understand its distribution and relationship with other features.

##### 2.4.1. Price Descriptive Statistics

Examine the central tendency and spread of rental prices.

In [ ]:
# Display price feature descriptive statistics
print(df['price'].describe())
print(f"\nPrice median: ${df['price'].median():.2f}")
print(f"Price standard deviation: ${df['price'].std():.2f}")

*The price distribution exhibits right skewness, indicating that while most listings have lower prices, a few high-priced properties significantly increase the mean. These extreme values may require outlier treatment in the data preparation phase.*

##### 2.4.2. Price Distribution

Visualize the overall distribution of rental prices.

In [ ]:
# Display price feature histogram 
df['price'].hist(bins=50, color=COLORS['primary'], figsize=(12, 5))
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.title('Price feature histogram')
plt.show()

##### 2.4.3. Price Distribution (Excluding Extreme Outliers)

Remove top 1% of prices to better visualize the typical price range.

In [ ]:
# Display price feature histogram without outliers
df[df['price'] < df['price'].quantile(0.99)]['price'].hist(bins=50, color=COLORS['primary'], figsize=(12, 5))
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.title('Price Distribution (99th percentile)')
plt.show()

*The majority of listings are priced below $5,000, with concentration in the lower price range.*

##### 2.4.4. Relationship Between Price and Property Characteristics

Analyze how property features (accommodates, bedrooms, beds, bathrooms) correlate with price.

In [ ]:
# Price vs key numeric features
key_features = ['accommodates', 'bedrooms', 'beds', 'bathrooms']

# Filter features that exist in dataframe
existing_features = [f for f in key_features if f in df.columns]

fig, axes = plt.subplots(1, len(existing_features), figsize=(18, 4))
axes = axes.flatten()

for idx, feature in enumerate(existing_features):
    if idx < len(axes):
        axes[idx].scatter(df[feature], df['price'], alpha=0.3, s=10, color=COLORS['primary'])
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel('Price')
        axes[idx].set_title(f'Price vs {feature}')

plt.tight_layout()
plt.show()

*While properties with more rooms were expected to command higher prices, the data shows considerable price variation regardless of room count, suggesting other factors also influence pricing.*

##### 2.4.5. Price Variation by Room Type and Property Type

Compare median prices across different accommodation categories.

In [ ]:
# Average price by room type
room_type_price = df.groupby('room_type')['price'].agg(['mean', 'median', 'count'])
print("Price statistics by Room Type:")
display(room_type_price.sort_values('median', ascending=False))
    
# Average price by property type (top 10)
property_type_price = df.groupby('property_type')['price'].agg(['mean', 'median', 'count'])
property_type_price = property_type_price.sort_values('median', ascending=False).head(10)
print("\nPrice statistics by Property Type (Top 10):")
display(property_type_price)
    
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
room_type_price['median'].plot(kind='barh', ax=axes[0], color=COLORS['primary'])
axes[0].set_title('Median Price by Room Type')
axes[0].set_xlabel('Median Price')
axes[0].set_ylabel('Room Type')
    
property_type_price['median'].plot(kind='barh', ax=axes[1], color=COLORS['primary'])
axes[1].set_title('Median Price by Property Type (Top 10)')
axes[1].set_xlabel('Median Price')
axes[1].set_ylabel('Property Type')
    
plt.tight_layout()
plt.show()

*Hotel rooms command the highest median prices among room types, which aligns with expectations. Notably, unique property types such as castles appear as outliers with significantly different pricing patterns compared to standard accommodation types.*

### 3. Data Preparation

This section addresses data cleaning, handling missing values, and transforming features to ensure the dataset is ready for modeling.

#### 3.1. Initial Missing Values Analysis

Identify and quantify missing values across all features before applying any transformations.

In [ ]:
# Calculate missing values for each column
missing_values = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})

# Filter columns with missing values and sort by percentage
missing_values = missing_values[missing_values['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)

print(f"Total columns with missing values: {len(missing_values)}")
print(f"\nColumns with >70% missing values: {len(missing_values[missing_values['Missing_Percentage'] > 70])}")
display(missing_values)

# Visualize missing values distribution
plt.figure(figsize=(12, 4))
plt.barh(missing_values['Column'][:10], missing_values['Missing_Percentage'][:10], color=COLORS['primary'])
plt.xlabel('Missing Percentage (%)')
plt.title('Top 10 Columns with Missing Values')
plt.axvline(x=70, color=COLORS['secondary'], linestyle='--', label='70% threshold')
plt.legend()
plt.tight_layout()
plt.show()

*The analysis reveals several columns with significant missing data. Columns exceeding the 70% threshold will be removed in the next step, as they provide insufficient information for model training.*

#### 3.2. Remove Irrelevant Features

Analyze and remove features that do not contribute to price prediction, including identifiers, URLs, text descriptions, and temporal fields.

In [ ]:
# Show all features and their top 5 common values
for col in df.columns:
    print(f"\nTop 5 common values for '{col}':")
    print(df[col].value_counts().head(5))

# Show DataFrame shape
print("Dataframe shape:", df.shape)

*Before applying transformations or removing inconsistent data, the following features will be removed as they do not impact rental price prediction at the time of listing publication. The objective is to predict price based on property amenities rather than external indicators: id, listing_url, scrape_id, last_scraped, source, name, description, neighborhood_overview, picture_url, host_id, host_url, host_name, host_since, host_location, host_about, host_response_time, host_response_rate, host_acceptance_rate, host_thumbnail_url, host_picture_url, host_neighbourhood, host_verifications, host_has_profile_pic, neighbourhood, neighbourhood_group_cleansed, latitude, longitude, bathrooms_text, minimum_minimum_nights, maximum_minimum_nights, minimum_maximum_nights, maximum_maximum_nights, minimum_nights_avg_ntm, maximum_nights_avg_ntm, calendar_updated, has_availability, availability_30, availability_60, availability_90, availability_365, calendar_last_scraped, number_of_reviews, number_of_reviews_ltm, number_of_reviews_l30d, availability_eoy, number_of_reviews_ly, estimated_occupancy_l365d, estimated_revenue_l365d, first_review, last_review, review_scores_rating, review_scores_accuracy, review_scores_cleanliness, review_scores_checkin, review_scores_communication, review_scores_location, review_scores_value, license, reviews_per_month*

In [ ]:
# Define features to remove
features_to_remove = [
    # IDS, URLs, sources and dates
    'id', 'scrape_id', 'last_scraped', 'source', 'listing_url', 'picture_url', 'calendar_last_scraped', 'calendar_updated',
    # Host information
    'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_verifications', 'host_has_profile_pic',
    # Occupancy and nights
    'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 
    # Text descriptions
    'name', 'description', 'neighborhood_overview', 'bathrooms_text',
    # License and availability
    'license', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'availability_eoy', 
    # Geographic
    'neighbourhood', 'neighbourhood_group_cleansed', 'latitude', 'longitude',
    # Reviews
    'first_review', 'last_review', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'reviews_per_month', 'number_of_reviews_ly', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
]

# Filter only existing columns
features_to_remove_existing = [col for col in features_to_remove if col in df.columns]

# Store original shape
original_shape = df.shape

# Remove features
df = df.drop(columns=features_to_remove_existing)

print(f"Original shape: {original_shape}")
print(f"Shape after removing irrelevant features: {df.shape}")
print(f"\nRemoved {len(features_to_remove_existing)} features:")
for feature in features_to_remove_existing:
    print(f"  - {feature}")

df.info()

*Successfully removed irrelevant features that do not contribute to price prediction. The dataset now focuses on property characteristics, location (neighbourhood), and amenities.*

*Since columns with >70% missing values have already been removed, no further action is required for this step.*

#### 3.3. Remove Inconsistent Records

Remove records with invalid or impossible values that would negatively impact model training.

In [ ]:
# Store original count
original_count = len(df)

# Define validation rules
print("Applying validation rules:")
print(f"Initial records: {len(df)}")

# Remove records with invalid price
invalid_price = df[df['price'].isnull() | (df['price'] <= 0)]
print(f" - Records with price <= 0 or NULL: {len(invalid_price)}")
df = df[(df['price'].notnull()) & (df['price'] > 0)]

# Remove records with invalid accommodates
invalid_accommodates = df[df['accommodates'].isnull() | (df['accommodates'] <= 0)]
print(f" - Records with accommodates <= 0 or NULL: {len(invalid_accommodates)}")
df = df[(df['accommodates'].notnull()) & (df['accommodates'] > 0)]

# Remove records with negative bedrooms, beds, or bathrooms
for col in ['bedrooms', 'beds', 'bathrooms']:
    if col in df.columns:
        invalid_count = len(df[df[col] < 0])
        if invalid_count > 0:
            print(f" - Records with {col} < 0: {invalid_count}")
            df = df[df[col] >= 0]

# Summary
records_removed = original_count - len(df)
print(f"\nTotal records removed: {records_removed}")
print(f"Remaining records: {len(df)}")
print(f"Percentage retained: {(len(df)/original_count)*100:.2f}%")

*Inconsistent and invalid records have been removed. The dataset now contains only listings with valid property characteristics suitable for price prediction.*

#### 3.4. Outlier Treatment

Address outliers in the target variable and key numeric features to improve model generalization.

##### 3.4.1. Remove Extreme Price Outliers

Remove listings with prices above the 99th percentile to focus on mainstream rental properties and eliminate extreme luxury outliers.

In [ ]:
# Calculate 99th percentile
price_99th = df['price'].quantile(0.99)

# Store original count
count_before = len(df)

# Filter outliers
df = df[df['price'] <= price_99th]

count_after = len(df)
outliers_removed = count_before - count_after

print(f"Price 99th percentile: ${price_99th:.2f}")
print(f"Records before filtering: {count_before}")
print(f"Records after filtering: {count_after}")
print(f"Outliers removed: {outliers_removed} ({(outliers_removed/count_before)*100:.2f}%)")
print(f"\nNew price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")
print(f"New price mean: ${df['price'].mean():.2f}")
print(f"New price median: ${df['price'].median():.2f}")

# Visualize price distribution after outlier removal
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df['price'], bins=50, color=COLORS['primary'])
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Price Distribution (After Outlier Removal)')

bp = axes[1].boxplot(df['price'], patch_artist=True)
bp['boxes'][0].set_facecolor(COLORS['primary'])
axes[1].set_ylabel('Price ($)')
axes[1].set_title('Price Boxplot (After Outlier Removal)')

plt.tight_layout()
plt.show()

*Extreme price outliers have been removed. The dataset now focuses on typical rental properties, which will improve model generalization for mainstream listings.*

##### 3.4.2. Cap Outliers in Numeric Features

Apply IQR method to cap outliers in key numeric features without removing records.

In [ ]:
# Define features to treat for outliers
numeric_features_for_outliers = ['accommodates', 'bedrooms', 'beds', 'bathrooms']

# Filter existing features
numeric_features_for_outliers = [f for f in numeric_features_for_outliers if f in df.columns]

# Function to cap outliers using IQR method
def cap_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count outliers before capping
    outliers_count = len(data[(data[column] < lower_bound) | (data[column] > upper_bound)])
    
    # Cap outliers
    data[column] = data[column].clip(lower=lower_bound, upper=upper_bound)
    
    return outliers_count, lower_bound, upper_bound

# Apply capping to each feature
print("Capping outliers using IQR method")
for feature in numeric_features_for_outliers:
    outliers, lower, upper = cap_outliers_iqr(df, feature)
    print(f"\n{feature}:")
    print(f" - Outliers capped: {outliers}")
    print(f" - Bounds: [{lower:.2f}, {upper:.2f}]")
    print(f" - New range: [{df[feature].min():.2f}, {df[feature].max():.2f}]")

*Outliers in numeric features have been capped using the IQR method. This approach retains all records while limiting the influence of extreme values on model training.*

#### 3.5. Fill Missing Values

Fill remaining missing values using appropriate strategies for numeric and categorical features.

##### 3.5.1. Analyze Remaining Missing Values

Identify features that still have missing values after previous cleaning steps.

In [ ]:
# Calculate missing values for remaining columns
remaining_missing = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100,
    'Data_Type': df.dtypes
})

# Filter only columns with missing values
remaining_missing = remaining_missing[remaining_missing['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)

print(f"Columns with remaining missing values: {len(remaining_missing)}\n")
display(remaining_missing)

# Visualize remaining missing values
plt.figure(figsize=(10, 7))
plt.barh(remaining_missing['Column'], remaining_missing['Missing_Percentage'], color=COLORS['primary'])
plt.xlabel('Missing Percentage (%)')
plt.title('Remaining Missing Values by Feature')
plt.tight_layout()
plt.show()

*Analysis shows the remaining features with missing values. Before imputation, feature selection will be performed to remove low-relevance features, followed by median imputation for numeric features and category assignment for categorical features.*

##### 3.5.2. Remove Redundant Features

Identify and remove features that provide overlapping information with other features.

In [ ]:
# Analyze correlation with target variable (price)
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
correlations = df[numeric_features].corr()['price'].sort_values(ascending=False)

print("Correlation analysis with price:")
print(correlations)

*Analysis reveals significant redundancy among host listing features. The features host_total_listings_count (r = 0.137) and calculated_host_listings_count (r = 0.136) show nearly identical correlation with price, differing by only 0.001. Additionally, calculated_host_listings_count appears to be the aggregate sum of the subcategory features (entire homes, shared rooms, and private rooms). Since the disaggregated features provide more granular information and calculated_host_listings_count_entire_homes exhibits the highest correlation (r = 0.167), the three redundant aggregate features will be removed while retaining the property-type-specific counts.*

In [ ]:
# Remove redundant host-related features
features_to_remove = [
    'host_listings_count',              # Lower correlation (0.098)
    'host_total_listings_count',        # Redundant aggregate (0.137)
    'calculated_host_listings_count'    # Redundant aggregate (0.136)
]

# Remove selected features
df = df.drop(columns=[col for col in features_to_remove if col in df.columns])
print(f"Dataset shape after feature removal: {df.shape}")

*Three redundant host-related features were successfully removed. The dataset now retains only the disaggregated property-type-specific listing counts, which provide more detailed information for price prediction.*

##### 3.5.3. Fill Numeric Features with Median

Replace missing values in numeric features with the median to maintain distribution robustness.

In [ ]:
# Identify numeric columns with missing values (after feature removal)
numeric_cols_with_missing = df.select_dtypes(include=[np.number]).columns[df.select_dtypes(include=[np.number]).isnull().any()].tolist()

print(f"Numeric columns with missing values: {len(numeric_cols_with_missing)}")

# Fill each numeric column with its median
for col in numeric_cols_with_missing:
    median_value = df[col].median()
    missing_count = df[col].isnull().sum()
    df[col].fillna(median_value, inplace=True)
    print(f" - {col}: Filled {missing_count} missing values with median - {median_value:.2f}")

*Median imputation was chosen over mean imputation due to the skewed distributions observed in the remaining features (|skewness| > 1), making the median more robust to outliers and better preserving the central tendency of each feature.*

##### 3.5.4. Fill Binary Categorical Features with Mode

Replace missing values in binary categorical features ('t' or 'f') with the most frequent value (mode).

In [ ]:
# Identify binary categorical columns ('t'/'f') with missing values
object_cols_with_missing = df.select_dtypes(include=['object']).columns[df.select_dtypes(include=['object']).isnull().any()].tolist()

# Filter only binary columns
binary_cols_with_missing = [col for col in object_cols_with_missing if set(df[col].dropna().unique()).issubset({'t', 'f'})]

print(f"Binary categorical columns with missing values: {len(binary_cols_with_missing)}\n")

# Show binary categorical columns with missing values
for feature in binary_cols_with_missing:
    missing_count = df[feature].isnull().sum()
    print(f"{feature}: {missing_count} missing values")

In [ ]:
# Fill binary categorical columns with mode
for col in binary_cols_with_missing:
    missing_count = df[col].isnull().sum()
    mode_value = df[col].mode()[0]
    
    df[col].fillna(mode_value, inplace=True)
    print(f"{col}: Filled {missing_count} missing values with mode: '{mode_value}'")

*Missing values in binary categorical features have been filled with the most frequent value, maintaining the original data distribution.*

##### 3.5.5. Convert Binary Categorical Features to Numeric

Convert binary categorical features from 't' or 'f' format to numeric format (1 or 0).

In [ ]:
# Identify all binary categorical columns ('t'/'f')
binary_cols = [col for col in df.select_dtypes(include=['object']).columns if set(df[col].unique()).issubset({'t', 'f'})]

print(f"Binary categorical columns to convert: {len(binary_cols)}\n")

# Show columns and their current distribution
for col in binary_cols:
    print(f"{col}: {df[col].value_counts().to_dict()}")

In [ ]:
# Convert 't'/'f' to 1/0
for col in binary_cols:
    # Remove literal quotes if present
    df[col] = df[col].str.strip("'")
    # Convert to numeric
    df[col] = (df[col] == 't').astype(int)
    print(f"{col}: {df[col].value_counts().to_dict()}")

*Binary categorical features have been converted from 't' or 'f' format to numeric format (1/0), where 1 represents True and 0 represents False.*

##### 3.5.6. Fill Non-Binary Categorical Features with New Category

Replace missing values in remaining categorical features with appropriate default categories.

In [ ]:
# Identify remaining categorical columns with missing values (excluding binary)
remaining_categorical_cols = df.select_dtypes(include=['object']).columns[df.select_dtypes(include=['object']).isnull().any()].tolist()

print(f"Remaining categorical columns with missing values: {len(remaining_categorical_cols)}\n")

# Show remaining categorical columns with missing values
for feature in remaining_categorical_cols:
    missing_count = df[feature].isnull().sum()
    print(f"{feature}: {missing_count} missing values")

In [ ]:
# If there are remaining categorical columns, fill them with 'Unknown'
if remaining_categorical_cols:
    # Fill each categorical column
    for col in remaining_categorical_cols:
        missing_count = df[col].isnull().sum()
        fill_value = 'Unknown'  # Default to 'Unknown'
        df[col].fillna(fill_value, inplace=True)
        print(f" - {col}: Filled {missing_count} missing values with: '{fill_value}'")
# Otherwise, confirm no remaining missing values
else:
    print("No remaining categorical columns with missing values.")

*All categorical features have been processed. Binary features were filled with mode, and other categorical features (if any) were filled with appropriate default categories.*

##### 3.5.7. Verify No Missing Values Remain

Confirm that all missing values have been addressed.

In [ ]:
# Check for any remaining missing values
total_missing = df.isnull().sum().sum()

print(f"Total missing values in dataset: {total_missing}")
print(f"Dataset shape: {df.shape}")

*All missing values have been successfully handled. The dataset is now complete and ready for further feature engineering.*

#### 3.6. Feature Engineering - Amenities

Transform the amenities list into binary features representing the most common amenities.

##### 3.6.1. Analyze Amenities Distribution

Identify the most frequent amenities across all listings.

In [ ]:
# Count all amenities
all_amenities = []
for amenities_str in df['amenities'].dropna():
    try:
        # Try to parse
        if isinstance(amenities_str, str):
            amenities_list = ast.literal_eval(amenities_str)
            all_amenities.extend(amenities_list)
    except:
        continue
    
# Create frequency distribution
amenities_counter = Counter(all_amenities)
top_amenities = amenities_counter.most_common(150)
    
# Display top 150 amenities
print("Top 150 Most Common Amenities:\n")
for idx, (amenity, count) in enumerate(top_amenities, 1):
    percentage = (count / len(df)) * 100
    print(f"{idx:2d}. {amenity:40s} - {count:5d} listings ({percentage:.1f}%)")
    
# Visualize
amenities_df = pd.DataFrame(top_amenities[:20], columns=['Amenity', 'Count'])
plt.figure(figsize=(12, 8))
plt.barh(amenities_df['Amenity'], amenities_df['Count'], color=COLORS['primary'])
plt.xlabel('Number of Listings')
plt.title('Top 20 Most Common Amenities (out of 150 selected)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

*The analysis reveals the most frequently offered amenities. These will be converted into binary features to capture their impact on rental prices*

##### 3.6.2 Create Binary Amenity Features

Create binary columns for the top 150 most common amenities.

In [ ]:
# Get list of filtered amenities (> 5%)
top_filtered_amenities = [amenity for amenity, count in top_amenities if count >= int(len(df) * 0.05)]

# Create dictionary to store all new columns
amenity_features = {}

# Create binary features for each top amenity
for amenity in top_filtered_amenities:
    # Create feature name (clean and lowercase)
    feature_name = f"has_{amenity.lower().replace(' ', '_').replace('-', '_')}"

    # Create binary column and store in dictionary
    amenity_features[feature_name] = df['amenities'].apply(lambda x: 1 if isinstance(x, str) and amenity in x else 0)
    
    # Count positive cases
    positive_count = amenity_features[feature_name].sum()
    print(f"{feature_name:50s} - {positive_count:5d} listings")

# Concatenate all new columns at once
amenity_features_df = pd.DataFrame(amenity_features)
df = pd.concat([df, amenity_features_df], axis=1)
    
print(f"\nTotal amenity features created: {len(top_filtered_amenities)}")
print(f"DataFrame shape after adding amenity features: {df.shape}")

*Binary amenity features have been created for amenities appearing in at least 5% of listings, allowing the model to learn their individual impact on pricing.*

##### 3.6.3 Create Amenities Count Feature

Create a feature representing the total number of amenities per listing.

In [ ]:
# Count total amenities for each listing
df['amenities_count'] = df['amenities'].apply(lambda x: len(ast.literal_eval(x)) if isinstance(x, str) and x != '' else 0)
    
# Display statistics
print("Amenities Count Statistics:")
print(df['amenities_count'].describe())
    
# Visualize distribution
plt.figure(figsize=(12, 5))
    
plt.subplot(1, 2, 1)
df['amenities_count'].hist(bins=30, color=COLORS['primary'])
plt.xlabel('Number of Amenities')
plt.ylabel('Frequency')
plt.title('Distribution of Amenities Count')
    
plt.subplot(1, 2, 2)
plt.scatter(df['amenities_count'], df['price'], alpha=0.3, s=10, color=COLORS['primary'])
plt.xlabel('Number of Amenities')
plt.ylabel('Price ($)')
plt.title('Price vs Amenities Count')
    
plt.tight_layout()
plt.show()

*A count feature has been created to capture the total number of amenities. This provides a simple numeric representation of listing completeness and quality.*

##### 3.6.4 Remove Original Amenities Column

Remove the original amenities text column as it has been transformed into structured features.

In [ ]:
df = df.drop(columns=['amenities'])
print(f"Current dataset shape: {df.shape}")

*The original amenities column has been removed after successful transformation into binary and count features.*

#### 3.7 Encode Categorical Features

Transform categorical variables into numeric format suitable for machine learning algorithms.

##### 3.7.1 Group Rare Categories in Property Type

Consolidate infrequent property types into an "Other" category to reduce dimensionality.

In [ ]:
# Calculate frequency of each property type
property_type_counts = df['property_type'].value_counts()
property_type_percentages = (property_type_counts / len(df)) * 100
    
print("Property Type Distribution (before grouping):")
print(f"Total unique property types: {len(property_type_counts)}\n")
display(pd.DataFrame({'Count': property_type_counts.head(15), 'Percentage': property_type_percentages.head(15)}))
    
# Define threshold for rare categories (less than 1%)
threshold_percentage = 1.0
rare_categories = property_type_percentages[property_type_percentages < threshold_percentage].index.tolist()
    
# Group rare categories into "Other"
df['property_type'] = df['property_type'].apply(lambda x: 'Other' if x in rare_categories else x)
print(f"Remaining unique property types: {df['property_type'].nunique()}")
    
# Show distribution after grouping
print("\nProperty Type Distribution (after grouping):")
display(df['property_type'].value_counts())

*Rare property types have been consolidated into an "Other" category, reducing dimensionality while retaining the most informative categories for the model.*

##### 3.7.2 One-Hot Encode Room Type

Apply one-hot encoding to room_type as it has few distinct categories.

In [ ]:
print("Room Type Distribution:")
print(df['room_type'].value_counts())

# Replace spaces with underscores in room_type values
df['room_type'] = df['room_type'].str.replace(' ', '_')
    
# Apply one-hot encoding
room_type_encoded = pd.get_dummies(df['room_type'], prefix='room_type', drop_first=True)
    
# Add encoded columns to dataframe
df = pd.concat([df, room_type_encoded], axis=1)
    
# Remove original column
df = df.drop(columns=['room_type'])

print(f"New features: {list(room_type_encoded.columns)}")

*Room type has been one-hot encoded into binary features. The first category is dropped to avoid multicollinearity (dummy variable trap).*

##### 3.7.3 One-Hot Encode Property Type

Apply one-hot encoding to property_type after grouping rare categories.

In [ ]:
print("Property Type Distribution:")
print(df['property_type'].value_counts())

# Replace spaces with underscores in property_type values
df['property_type'] = df['property_type'].str.replace(' ', '_')
    
# Apply one-hot encoding
property_type_encoded = pd.get_dummies(df['property_type'], prefix='property_type', drop_first=True)
    
# Add encoded columns to dataframe
df = pd.concat([df, property_type_encoded], axis=1)
    
# Remove original column
df = df.drop(columns=['property_type'])

print(f"New features: {list(property_type_encoded.columns)}")

*Property type has been one-hot encoded after consolidating rare categories, creating binary features that represent property classification.*

##### 3.7.4 Label Encode Neighbourhood

Apply label encoding to neighbourhood_cleansed due to the high number of unique neighbourhoods.

In [ ]:
print(f"Unique neighbourhoods: {df['neighbourhood_cleansed'].nunique()}")
print("\nNeighbourhood distribution:")
display(df['neighbourhood_cleansed'].value_counts())
    
# Apply label encoding
le = LabelEncoder()
df['neighbourhood_encoded'] = le.fit_transform(df['neighbourhood_cleansed'])
    
# Store mapping for reference
neighbourhood_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    
print(f"Encoding range: {df['neighbourhood_encoded'].min()} to {df['neighbourhood_encoded'].max()}")
    
# Remove original column
df = df.drop(columns=['neighbourhood_cleansed'])

*Neighbourhood has been label encoded into numeric values, preserving all neighbourhood information while reducing memory usage compared to one-hot encoding.*

#### 3.8 Feature Selection and Correlation Analysis

Analyze feature correlations and remove features with weak relationships to the target variable.

##### 3.8.1 Calculate Correlation with Price

Compute correlation coefficients between all numeric features and the target variable.

In [ ]:
# Calculate correlation with price for all numeric features
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove price from features list
numeric_features.remove('price')

# Filter out features with zero standard deviation (constant features)
valid_features = []
for feature in numeric_features:
    if df[feature].std() > 0:  # Only keep features with variation
        valid_features.append(feature)

# Calculate correlations only for valid features
correlations = df[valid_features].corrwith(df['price']).sort_values(ascending=False)
    
print(f"Total numeric features analyzed: {len(correlations)}\n")
print("Top 15 Features - Highest Positive Correlation with Price:")
display(correlations.head(15))
    
print("\nTop 15 Features - Highest Negative Correlation with Price:")
display(correlations.tail(15))
    
# Visualize correlations
plt.figure(figsize=(12, 6))
pd.concat([correlations.head(15), correlations.tail(15)]).plot(kind='barh', color=COLORS['primary'])
plt.xlabel('Correlation with Price')
plt.title('Feature Correlation with Price')
plt.axvline(x=0.05, color=COLORS['secondary'], linestyle='--', alpha=0.5, label='|r| = 0.05 threshold')
plt.axvline(x=-0.05, color=COLORS['secondary'], linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

*Correlation analysis reveals which features have the strongest linear relationships with price. Features with very weak correlations may be candidates for removal.*

##### 3.8.2 Remove Low Correlation Features

Remove features with correlation coefficient below the threshold (|r| < 0.05).

In [ ]:
# Define correlation threshold
correlation_threshold = 0.05

# Identify features with low correlation
low_correlation_features = correlations[abs(correlations) < correlation_threshold].index.tolist()

# Show features with weak correlation to be removed
print(f"Features with |correlation| < {correlation_threshold}: {len(low_correlation_features)}")
print("\nFeatures to be removed:")
for feature in low_correlation_features:
    print(f"  - {feature}: {correlations[feature]:.4f}")
    
# Store shape before removal
shape_before = df.shape
    
# Remove low correlation features
df = df.drop(columns=low_correlation_features)
    
print(f"\nRemoved {len(low_correlation_features)} low correlation features")
print(f"Shape before: {shape_before}")
print(f"Shape after: {df.shape}")

*Features with weak correlation to price have been removed, focusing the model on the most predictive variables and reducing noise.*

#### 3.9 Data Shuffling

Randomize the order of records to prevent any ordering bias during model training.

In [ ]:
# Set random seed for reproducibility
random_state = 42

# Shuffle the dataset
df_shuffled = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

# Update the main dataframe
df = df_shuffled

print(f"Dataset shuffled with random_state={random_state}")
print(f"Dataset shape: {df.shape}")

*The dataset has been randomly shuffled to ensure that any inherent ordering in the data does not bias the train-test split or model training process.*

#### 3.10 Train-Test Split

Divide the dataset into training and testing sets for model development and evaluation.

In [ ]:
# Separate features and target
X = df.drop(columns=['price'])
y = df['price']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display train-test split results
print("Train-Test Split Results:")
print(f"Training set size: {len(X_train)} samples ({(len(X_train)/len(df))*100:.1f}%)")
print(f"Testing set size: {len(X_test)} samples ({(len(X_test)/len(df))*100:.1f}%)")
print(f"\nFeatures shape: {X_train.shape}")
print(f"\nTarget statistics:")
print(f" Train - Mean: ${y_train.mean():.2f}, Median: ${y_train.median():.2f}")
print(f" Test  - Mean: ${y_test.mean():.2f}, Median: ${y_test.median():.2f}")

*The dataset has been split into training (80%) and testing (20%) sets. Both sets maintain similar price distributions, ensuring fair model evaluation.*

#### 3.11 Feature Scaling

Standardize numeric features to have zero mean and unit variance for improved model performance.

In [ ]:
# Initialize scaler
scaler = StandardScaler()

# Identify numeric features (excluding binary amenity features)
numeric_features_to_scale = []
for col in X_train.columns:
    # Scale only continuous numeric features, not binary features
    if X_train[col].dtype in ['int64', 'float64']:
    # Check if feature is not binary (has more than 2 unique values)
        if X_train[col].nunique() > 2:
            numeric_features_to_scale.append(col)

print(f"Features to be scaled: {len(numeric_features_to_scale)}")
print(f"Features: {numeric_features_to_scale}")

# Fit scaler on training data only
scaler.fit(X_train[numeric_features_to_scale])

# Transform both training and testing data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features_to_scale] = scaler.transform(X_train[numeric_features_to_scale])
X_test_scaled[numeric_features_to_scale] = scaler.transform(X_test[numeric_features_to_scale])

# Display sample statistics after scaling
print("\nSample statistics after scaling (training set):")
display(X_train_scaled[numeric_features_to_scale].describe())

*Numeric features have been standardized using StandardScaler fitted on training data only. This prevents data leakage and ensures consistent scaling for model predictions.*

#### 3.12 Final Data Preparation Summary

Review the final prepared datasets ready for model training.

In [ ]:
print("SUMMARY AFTER DATA PREPARATION")

print("\n1. DATASET OVERVIEW:")
print(f"Total samples: {len(df)}")
print(f"Training samples: {len(X_train_scaled)}")
print(f"Testing samples: {len(X_test_scaled)}")
print(f"Total features: {X_train_scaled.shape[1]}")

print("\n2. FEATURE COMPOSITION:")
feature_types = X_train_scaled.dtypes.value_counts()
print(feature_types)

print("\n3. TARGET VARIABLE (PRICE) STATISTICS:")
print(f"Training set:")
print(f" - Mean: ${y_train.mean():.2f}")
print(f" - Median: ${y_train.median():.2f}")
print(f" - Std Dev: ${y_train.std():.2f}")
print(f" - Range: ${y_train.min():.2f} - ${y_train.max():.2f}")

print(f"Testing set:")
print(f" - Mean: ${y_test.mean():.2f}")
print(f" - Median: ${y_test.median():.2f}")
print(f" - Std Dev: ${y_test.std():.2f}")
print(f" - Range: ${y_test.min():.2f} - ${y_test.max():.2f}")

print("\n4. DATA QUALITY:")
print(f"Missing values in training set: {X_train_scaled.isnull().sum().sum()}")
print(f"Missing values in testing set: {X_test_scaled.isnull().sum().sum()}")
print(f"Missing values in target (train): {y_train.isnull().sum()}")
print(f"Missing values in target (test): {y_test.isnull().sum()}")

*Data preparation is complete. The dataset has been cleaned, transformed, and split into training and testing sets. All features are numeric and scaled, with no missing values. The data is now ready for machine learning model training and evaluation.*

### 4. Model Development

This section trains and optimizes machine learning models to predict Airbnb rental prices. Models are evaluated with default parameters, compared on key metrics, tuned through grid search, and the best-performing model is selected.

#### 4.1. Candidate Models Overview

The following models will be trained and compared:
- **Linear Regression**: Simple baseline model assuming linear relationships
- **Random Forest Regressor**: Ensemble method capable of capturing non-linear patterns

*Detailed rationale for model selection, including consideration of alternative algorithms (Gradient Boosting, XGBoost), is provided in the accompanying report.*

#### 4.2. Model Training with Default Parameters

Train both models using scikit-learn's default hyperparameters to establish baseline performance.

##### 4.2.1. Define Model Evaluation Function

Create a reusable function to evaluate regression model performance using standard metrics.

In [ ]:
# Define evaluation metrics
def evaluate_model(y_true, y_pred, model_name):
    """
    Calculate and display regression evaluation metrics.
    
    Parameters:
    - y_true: actual target values
    - y_pred: predicted target values
    - model_name: name of the model for display
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"{model_name} Performance:")
    print(f" - RMSE (Root Mean Squared Error): ${rmse:.2f}")
    print(f" - MAE (Mean Absolute Error): ${mae:.2f}")
    print(f" - R² Score: {r2:.4f}")
    
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2}

print("Evaluation Metrics Defined:")
print(" - RMSE: Measures average prediction error magnitude (lower is better)")
print(" - MAE: Measures average absolute prediction error (lower is better)")
print(" - R²: Proportion of variance explained by the model (higher is better, max = 1.0)")

*Evaluation function defined with three key metrics: RMSE and MAE measure prediction accuracy in dollar terms, while R² indicates the proportion of variance explained by the model.*

##### 4.2.2. Train Linear Regression Model

Train a Linear Regression model as a simple baseline.

In [ ]:
# Initialize and train Linear Regression model
lr_model = LinearRegression()

# Record training time
start_time = time.time()
lr_model.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

# Display training summary
print(f"Training time: {training_time:.2f} seconds")
print(f"Number of features: {X_train_scaled.shape[1]}")
print(f"Coefficients computed: {len(lr_model.coef_)}")

*Linear Regression model trained using Ordinary Least Squares (OLS) method. The model assumes linear relationships between features and price.*

##### 4.2.3. Evaluate Linear Regression - Training Set

Assess model performance on training data to check for underfitting.

In [ ]:
# Make predictions on training set
y_train_pred_lr = lr_model.predict(X_train_scaled)

# Evaluate on training set
train_metrics_lr = evaluate_model(y_train, y_train_pred_lr, "Linear Regression (Train)")

##### 4.2.4. Evaluate Linear Regression - Test Set

Evaluate generalization performance on unseen test data.

In [ ]:
# Make predictions on test set
y_test_pred_lr = lr_model.predict(X_test_scaled)

# Evaluate on test set
test_metrics_lr = evaluate_model(y_test, y_test_pred_lr, "Linear Regression (Test)")

# Check for overfitting
print(f"\nOverfitting Analysis:")
print(f"Train R²: {train_metrics_lr['R2']:.4f}")
print(f"Test R²: {test_metrics_lr['R2']:.4f}")
print(f"Difference: {abs(train_metrics_lr['R2'] - test_metrics_lr['R2']):.4f}")

*Linear Regression shows similar performance on training and test sets, indicating no overfitting. However, the R² score suggests limited ability to capture complex pricing patterns.*

##### 4.2.5. Train Random Forest Model

Train a Random Forest Regressor with default parameters.

In [ ]:
# Initialize Random Forest with default parameters
rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)

# Record training time
start_time = time.time()
rf_model.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

print(f"Training time: {training_time:.2f} seconds")
print(f"Number of trees: {rf_model.n_estimators}")
print(f"Number of features used: {rf_model.n_features_in_}")

*Random Forest uses bootstrap aggregating (bagging) to train multiple decision trees on random subsets of data and features. The default configuration uses 100 trees without depth limitation.*

##### 4.2.6. Evaluate Random Forest - Training Set

Assess Random Forest performance on training data.

In [ ]:
# Make predictions on training set
y_train_pred_rf = rf_model.predict(X_train_scaled)

# Evaluate on training set
train_metrics_rf = evaluate_model(y_train, y_train_pred_rf, "Random Forest (Train)")

##### 4.2.7. Evaluate Random Forest - Test Set

Evaluate Random Forest generalization on test data.

In [ ]:
# Make predictions on test set
y_test_pred_rf = rf_model.predict(X_test_scaled)

# Evaluate on test set
test_metrics_rf = evaluate_model(y_test, y_test_pred_rf, "Random Forest (Test)")

# Check for overfitting
print(f"\nOverfitting Analysis:")
print(f"Train R²: {train_metrics_rf['R2']:.4f}")
print(f"Test R²: {test_metrics_rf['R2']:.4f}")
print(f"Difference: {abs(train_metrics_rf['R2'] - test_metrics_rf['R2']):.4f}")

if abs(train_metrics_rf['R2'] - test_metrics_rf['R2']) > 0.1:
    print("\nPotential overfitting detected (R² difference > 0.1)")
else:
    print("\nNo significant overfitting detected")

*Random Forest typically shows higher training performance than test performance due to its ensemble nature. The gap between training and test R² indicates the model's tendency to overfit.*

#### 4.3. Initial Model Comparison

Compare baseline performance of both models before hyperparameter tuning.

In [ ]:
# Create comprehensive comparison DataFrame
comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'Train_R2': [train_metrics_lr['R2'], train_metrics_rf['R2']],
    'Test_R2': [test_metrics_lr['R2'], test_metrics_rf['R2']],
    'Train_RMSE': [train_metrics_lr['RMSE'], train_metrics_rf['RMSE']],
    'Test_RMSE': [test_metrics_lr['RMSE'], test_metrics_rf['RMSE']],
    'Train_MAE': [train_metrics_lr['MAE'], train_metrics_rf['MAE']],
    'Test_MAE': [test_metrics_lr['MAE'], test_metrics_rf['MAE']],
    'Overfitting_Gap': [
        abs(train_metrics_lr['R2'] - test_metrics_lr['R2']),
        abs(train_metrics_rf['R2'] - test_metrics_rf['R2'])
    ]
})

print("\nModel Comparison Summary")
display(comparison_df)

# Visualize comparison with train and test metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = np.arange(len(comparison_df['Model']))
width = 0.35

# R² Comparison
r2_train = axes[0].bar(x - width/2, comparison_df['Train_R2'], width, color=COLORS['primary'])
r2_test = axes[0].bar(x + width/2, comparison_df['Test_R2'], width, color=COLORS['secondary'])
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Score Comparison (Train vs Test)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df['Model'])
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', color=COLORS['bg'], linestyle='--')

# Add value labels
for i, (train_val, test_val) in enumerate(zip(comparison_df['Train_R2'], comparison_df['Test_R2'])):
    axes[0].text(i - width/2, train_val + 0.02, f'{train_val:.4f}', ha='center')
    axes[0].text(i + width/2, test_val + 0.02, f'{test_val:.4f}', ha='center')

# RMSE Comparison
rmse_train = axes[1].bar(x - width/2, comparison_df['Train_RMSE'], width, color=COLORS['primary'])
rmse_test = axes[1].bar(x + width/2, comparison_df['Test_RMSE'], width, color=COLORS['secondary'])
axes[1].set_ylabel('RMSE ($)')
axes[1].set_title('RMSE Comparison - Lower is Better')
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison_df['Model'])
axes[1].grid(axis='y', color=COLORS['bg'], linestyle='--')

# Add value labels
for i, (train_val, test_val) in enumerate(zip(comparison_df['Train_RMSE'], comparison_df['Test_RMSE'])):
    axes[1].text(i - width/2, train_val + max(comparison_df['Train_RMSE'])*0.02, f'${train_val:.2f}', ha='center')
    axes[1].text(i + width/2, test_val + max(comparison_df['Test_RMSE'])*0.02, f'${test_val:.2f}', ha='center')

# MAE Comparison
mae_train = axes[2].bar(x - width/2, comparison_df['Train_MAE'], width, color=COLORS['primary'])
mae_test = axes[2].bar(x + width/2, comparison_df['Test_MAE'], width, color=COLORS['secondary'])
axes[2].set_ylabel('MAE ($)')
axes[2].set_title('MAE Comparison - Lower is Better')
axes[2].set_xticks(x)
axes[2].set_xticklabels(comparison_df['Model'])
axes[2].grid(axis='y', color=COLORS['bg'], linestyle='--')

# Add value labels
for i, (train_val, test_val) in enumerate(zip(comparison_df['Train_MAE'], comparison_df['Test_MAE'])):
    axes[2].text(i - width/2, train_val + max(comparison_df['Train_MAE'])*0.02, f'${train_val:.2f}', ha='center')
    axes[2].text(i + width/2, test_val + max(comparison_df['Test_MAE'])*0.02, f'${test_val:.2f}', ha='center')

# Add single shared legend
fig.legend(['Train', 'Test'], loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=True)

plt.tight_layout()
plt.show()

#### 4.4. Hyperparameter Tuning

Optimize model performance through systematic hyperparameter tuning.

##### 4.4.1. Random Forest Hyperparameter Tuning

Apply Grid Search to find optimal Random Forest parameters, focusing on overfitting mitigation.

In [ ]:
print("Hyperparameter Tuning Strategy:")
print("- Method: Randomized Search with 3-fold Cross-Validation")
print("- Focus: Balance performance and overfitting")
print("- Regularization: max_depth, min_samples_split, min_samples_leaf")

# Define parameter grid (focused on overfitting control)
param_grid = {
    'n_estimators': [100, 200],     # Number of trees
    'max_depth': [10, 20, None],    # Tree depth limit (regularization)
    'min_samples_split': [2, 10],   # Minimum samples to split (regularization)
    'min_samples_leaf': [1, 4],     # Minimum samples per leaf (regularization)
    'max_features': ['sqrt']        # Features per split
}

print(f"\nTotal combinations: {2 * 3 * 2 * 2 * 1}")
print("Estimated time: ~1-3 minutes\n")

# Initialize Grid Search
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,                          # 3-fold cross-validation
    scoring='r2',                  # Optimize for R²
    verbose=2,
    n_jobs=-1,
)

# Perform Grid Search
start_time = time.time()
grid_search.fit(X_train_scaled, y_train)
tuning_time = time.time() - start_time

print(f"Grid Search completed in {tuning_time/60:.2f} minutes")

*Grid Search exhaustively tests all 24 hyperparameter combinations using 3-fold cross-validation, focusing on regularization parameters to balance model complexity and prevent overfitting.*

##### 4.4.2. Best Parameters and Model Retraining

Identify optimal parameters and retrain the final Random Forest model.

In [ ]:
# Display best parameters
print("Best Parameters Found:")
for param, value in grid_search.best_params_.items():
    print(f" - {param}: {value}")

print(f"Best Cross-Validation R² Score: {grid_search.best_score_:.4f}")

# Retrain with best parameters
rf_model_tuned = grid_search.best_estimator_
print()

# Evaluate tuned model on training set
y_train_pred_rf_tuned = rf_model_tuned.predict(X_train_scaled)
train_metrics_rf_tuned = evaluate_model(y_train, y_train_pred_rf_tuned, "Random Forest Tuned (Train)")
print()

# Evaluate tuned model on test set
y_test_pred_rf_tuned = rf_model_tuned.predict(X_test_scaled)
test_metrics_rf_tuned = evaluate_model(y_test, y_test_pred_rf_tuned, "Random Forest Tuned (Test)")

# Overfitting analysis
print(f"\nOverfitting Analysis (Tuned Model):")
print(f"Train R²: {train_metrics_rf_tuned['R2']:.4f}")
print(f"Test R²: {test_metrics_rf_tuned['R2']:.4f}")
print(f"Difference: {abs(train_metrics_rf_tuned['R2'] - test_metrics_rf_tuned['R2']):.4f}")

*The tuned Random Forest model incorporates regularization techniques through controlled tree depth and minimum sample requirements. These constraints reduce model complexity and improve generalization to unseen data.*

##### 4.4.3. Compare Default vs Tuned Random Forest

Evaluate the impact of hyperparameter tuning on model performance.

In [ ]:
# Create comprehensive comparison DataFrame
tuning_comparison = pd.DataFrame({
    'Configuration': ['Default', 'Tuned'],
    'Train_R2': [train_metrics_rf['R2'], train_metrics_rf_tuned['R2']],
    'Test_R2': [test_metrics_rf['R2'], test_metrics_rf_tuned['R2']],
    'Train_RMSE': [train_metrics_rf['RMSE'], train_metrics_rf_tuned['RMSE']],
    'Test_RMSE': [test_metrics_rf['RMSE'], test_metrics_rf_tuned['RMSE']],
    'Train_MAE': [train_metrics_rf['MAE'], train_metrics_rf_tuned['MAE']],
    'Test_MAE': [test_metrics_rf['MAE'], test_metrics_rf_tuned['MAE']],
    'Overfitting_Gap': [
        abs(train_metrics_rf['R2'] - test_metrics_rf['R2']),
        abs(train_metrics_rf_tuned['R2'] - test_metrics_rf_tuned['R2'])
    ]
})

print("\nDefault vs Tuned Random Forest Comparison")
display(tuning_comparison)

# Visualize comparison with train and test metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = np.arange(len(tuning_comparison['Configuration']))
width = 0.35

# R² Comparison
r2_train = axes[0].bar(x - width/2, tuning_comparison['Train_R2'], width, color=COLORS['primary'])
r2_test = axes[0].bar(x + width/2, tuning_comparison['Test_R2'], width, color=COLORS['secondary'])
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Score Comparison (Train vs Test)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tuning_comparison['Configuration'])
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', color=COLORS['bg'], linestyle='--')

# Add value labels
for i, (train_val, test_val) in enumerate(zip(tuning_comparison['Train_R2'], tuning_comparison['Test_R2'])):
    axes[0].text(i - width/2, train_val + 0.02, f'{train_val:.4f}', ha='center')
    axes[0].text(i + width/2, test_val + 0.02, f'{test_val:.4f}', ha='center')

# RMSE Comparison
rmse_train = axes[1].bar(x - width/2, tuning_comparison['Train_RMSE'], width, color=COLORS['primary'])
rmse_test = axes[1].bar(x + width/2, tuning_comparison['Test_RMSE'], width, color=COLORS['secondary'])
axes[1].set_ylabel('RMSE ($)')
axes[1].set_title('RMSE Comparison - Lower is Better')
axes[1].set_xticks(x)
axes[1].set_xticklabels(tuning_comparison['Configuration'])
axes[1].grid(axis='y', color=COLORS['bg'], linestyle='--')

# Add value labels
for i, (train_val, test_val) in enumerate(zip(tuning_comparison['Train_RMSE'], tuning_comparison['Test_RMSE'])):
    axes[1].text(i - width/2, train_val + max(tuning_comparison['Train_RMSE'])*0.02, f'${train_val:.2f}', ha='center')
    axes[1].text(i + width/2, test_val + max(tuning_comparison['Test_RMSE'])*0.02, f'${test_val:.2f}', ha='center')

# MAE Comparison
mae_train = axes[2].bar(x - width/2, tuning_comparison['Train_MAE'], width, color=COLORS['primary'])
mae_test = axes[2].bar(x + width/2, tuning_comparison['Test_MAE'], width, color=COLORS['secondary'])
axes[2].set_ylabel('MAE ($)')
axes[2].set_title('MAE Comparison - Lower is Better')
axes[2].set_xticks(x)
axes[2].set_xticklabels(tuning_comparison['Configuration'])
axes[2].grid(axis='y', color=COLORS['bg'], linestyle='--')

# Add value labels
for i, (train_val, test_val) in enumerate(zip(tuning_comparison['Train_MAE'], tuning_comparison['Test_MAE'])):
    axes[2].text(i - width/2, train_val + max(tuning_comparison['Train_MAE'])*0.02, f'${train_val:.2f}', ha='center')
    axes[2].text(i + width/2, test_val + max(tuning_comparison['Test_MAE'])*0.02, f'${test_val:.2f}', ha='center')

# Add single shared legend
fig.legend(['Train', 'Test'], loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=True)

plt.tight_layout()
plt.show()

# Print improvement summary
r2_improvement = ((tuning_comparison.loc[1, 'Test_R2'] - tuning_comparison.loc[0, 'Test_R2']) / tuning_comparison.loc[0, 'Test_R2']) * 100
rmse_improvement = ((tuning_comparison.loc[0, 'Test_RMSE'] - tuning_comparison.loc[1, 'Test_RMSE']) / tuning_comparison.loc[0, 'Test_RMSE']) * 100
overfitting_reduction = ((tuning_comparison.loc[0, 'Overfitting_Gap'] - tuning_comparison.loc[1, 'Overfitting_Gap']) / tuning_comparison.loc[0, 'Overfitting_Gap']) * 100

print(f"Test R² improvement: {r2_improvement:+.2f}%")
print(f"Test RMSE improvement: {rmse_improvement:+.2f}%")
print(f"Overfitting gap reduction: {overfitting_reduction:+.2f}%")

*Hyperparameter tuning successfully reduces overfitting while maintaining or improving test set performance. The regularization techniques (tree depth limitation and sample requirements) create a more generalizable model.*

#### 4.5 Final Model Selection

Select the best-performing model for deployment based on comprehensive evaluation.

In [ ]:
# Create final comparison
final_comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (Default)', 'Random Forest (Tuned)'],
    'Test_R2': [test_metrics_lr['R2'], test_metrics_rf['R2'], test_metrics_rf_tuned['R2']],
    'Test_RMSE': [test_metrics_lr['RMSE'], test_metrics_rf['RMSE'], test_metrics_rf_tuned['RMSE']],
    'Test_MAE': [test_metrics_lr['MAE'], test_metrics_rf['MAE'], test_metrics_rf_tuned['MAE']],
    'Overfitting_Gap': [
        abs(train_metrics_lr['R2'] - test_metrics_lr['R2']),
        abs(train_metrics_rf['R2'] - test_metrics_rf['R2']),
        abs(train_metrics_rf_tuned['R2'] - test_metrics_rf_tuned['R2'])
    ]
})

print("FINAL MODEL COMPARISON")
display(final_comparison)

# Identify best model
best_model_idx = final_comparison['Test_R2'].idxmax()
best_model_name = final_comparison.loc[best_model_idx, 'Model']

print(f"Selected model: {best_model_name}")
print(f"Performance Metrics:")
print(f" - Test R²: {final_comparison.loc[best_model_idx, 'Test_R2']:.4f}")
print(f" - Test RMSE: ${final_comparison.loc[best_model_idx, 'Test_RMSE']:.2f}")
print(f" - Test MAE: ${final_comparison.loc[best_model_idx, 'Test_MAE']:.2f}")
print(f" - Overfitting Gap: {final_comparison.loc[best_model_idx, 'Overfitting_Gap']:.4f}")

*The **Random Forest (Tuned)** model is selected as the final model for Airbnb price prediction. It achieves the best balance between predictive accuracy and generalization, with effective overfitting mitigation through hyperparameter optimization.*